# Release16 Joint Target Holdout Slice Repair

This notebook reconciles the latest post-short-training checkpoint blocker with the existing fold-local joint target contract. It separates target-side readiness from holdout slice generalization so the next retained-model loop does not run PPO/checkpoint replay from a train-only signal.

Governance fields:
- Baselines: canonical `soxlbacktest_qlib_walkforward_baseline_v1`, direct comparison `release16_post_short_training_reward_checkpoint_contract_v1`.
- Role: `diagnose`.
- Target window: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Changed layer: `target`, `model_selection`, `bc_to_ppo_handoff`.
- Hypothesis: the fold-local joint target remains non-leaking and label-ready, but the selected BC/reward checkpoint handoff is blocked by holdout slice failure, so training must stay disabled until state-conditioned action/loss repair is proven on holdout slices.


In [1]:
from __future__ import annotations

import json
import math
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_objective_training_targets import (
    audit_state_objective_target_lift,
    audit_state_objective_target_scope,
    score_state_objective_bc_readiness,
    summarize_state_objective_bc_readiness,
    summarize_state_objective_target_lift_audit,
    summarize_state_objective_target_scope_audit,
)

EXPERIMENT = "release16_joint_target_holdout_slice_repair_v1"
OUT_DIR = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_TARGET_CARD = ROOT / "runtime" / "model_selection" / "card" / "release16_joint_drawdown_release_target_contract_v1"
SOURCE_HOLDOUT_CARD = ROOT / "runtime" / "model_selection" / "card" / "release16_joint_drawdown_aux_state_head_bc_holdout_replay_v1"
POST_SHORT_CARD = ROOT / "runtime" / "model_selection" / "card" / "release16_post_short_training_reward_checkpoint_contract_v1"
SHORT_SCREEN_CARD = ROOT / "runtime" / "model_selection" / "card" / "release16_drawdown_drift_short_training_screen_v1"

REQUIRED_FOLDS = ["wf_2020", "wf_2022", "wf_2023_2024"]
TARGET_MAX_DD = 20.0
TARGET_RELATIVE_RETURN = 30.0
MIN_SLICE_ACCURACY = 0.60


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def fold_eval_name(fold: str) -> str:
    return str(fold).removesuffix("_train")


def action_entropy(actions: pd.Series) -> float:
    counts = actions.dropna().astype(str).value_counts(normalize=True)
    return float(-(counts * counts.map(math.log)).sum()) if not counts.empty else 0.0


def label_distribution(frame: pd.DataFrame, *, profile: str) -> dict[str, object]:
    actions = frame["action"].dropna().astype(str) if "action" in frame.columns else pd.Series(dtype=str)
    counts = actions.value_counts(normalize=True)
    return {
        "profile": profile,
        "samples": int(len(actions)),
        "unique_actions": int(actions.nunique()),
        "dominant_action_share": float(counts.max()) if not counts.empty else 1.0,
        "action_entropy": action_entropy(actions),
    }


def bool_all(values: pd.Series) -> bool:
    return bool(values.fillna(False).astype(bool).all()) if len(values) else False


In [2]:
source_summary = read_json(SOURCE_TARGET_CARD / "summary.json")
source_holdout_summary = read_json(SOURCE_HOLDOUT_CARD / "summary.json")
post_short_summary = read_json(POST_SHORT_CARD / "summary.json")

joint = pd.read_csv(SOURCE_TARGET_CARD / "joint_target_contract.csv", parse_dates=["datetime"])
strict = pd.read_csv(SOURCE_TARGET_CARD / "strict_joint_target_contract.csv", parse_dates=["datetime"])
source_bc_scores = pd.read_csv(SOURCE_TARGET_CARD / "bc_readiness_scores.csv")
latest_holdout = pd.read_csv(POST_SHORT_CARD / "bc_holdout_slice_readout.csv")
latest_train_holdout = pd.read_csv(SHORT_SCREEN_CARD / "bc_slice_metrics.csv")
source_holdout_slices = pd.read_csv(SOURCE_HOLDOUT_CARD / "bc_slice_metrics.csv")

fold_rows = (
    joint[["fold", "train_start", "train_end", "eval_start", "eval_end"]]
    .drop_duplicates()
    .sort_values("fold")
)
folds = [
    {
        "name": str(row.fold),
        "train_start": str(row.train_start),
        "train_end": str(row.train_end),
        "eval_start": str(row.eval_start),
        "eval_end": str(row.eval_end),
    }
    for row in fold_rows.itertuples(index=False)
]

scope_audit = audit_state_objective_target_scope(joint, folds=folds)
scope_summary = summarize_state_objective_target_scope_audit(scope_audit)

lift_audit = audit_state_objective_target_lift(
    joint,
    group_cols=("fold",),
    baseline_risk_position_col="risk_target_position_original",
)
lift_summary = summarize_state_objective_target_lift_audit(lift_audit)

# The source contract's BC readiness is based on the target/effective action surface.
# The raw `action` column in the contract is a passthrough trading action and is
# intentionally not used as the BC label distribution here.
bc_readiness = source_bc_scores.copy()
bc_train_summary = summarize_state_objective_bc_readiness(bc_readiness)

scope_audit.to_csv(OUT_DIR / "target_scope_audit.csv", index=False)
lift_audit.to_csv(OUT_DIR / "target_lift_audit.csv", index=False)
bc_readiness.to_csv(OUT_DIR / "bc_readiness_scores.csv", index=False)

pd.DataFrame([
    {"gate": "target_scope", "passed": bool(scope_summary["training_allowed"]), **scope_summary},
    {"gate": "target_lift", "passed": bool(lift_summary["training_allowed"]), **lift_summary},
    {"gate": "bc_train_label_readiness", "passed": bool(bc_train_summary["selected_bc_readiness_gate"]), **bc_train_summary},
]).to_csv(OUT_DIR / "target_contract_gate.csv", index=False)

print("scope", scope_summary)
print("lift", lift_summary)
print("bc_train", bc_train_summary)


scope {'decision': 'state_objective_targets_training_ready', 'training_allowed': True, 'leakage_gate': True, 'audited_groups': 3, 'blocked_groups': 0, 'eval_overlap_rows': 0, 'outside_train_eval_rows': 0, 'scope_issues': [], 'top_scope_issue': '', 'next_mechanism': 'run_fold_local_state_objective_training_or_bc'}
lift {'decision': 'state_objective_target_lift_training_ready', 'training_allowed': True, 'lift_semantics_gate': True, 'audited_groups': 3, 'blocked_groups': 0, 'rebound_rows': 1903, 'current_positive_rebound_lift_rows': 1376, 'candidate_release_positive_lift_rows': 1901, 'rebound_downtrend_rows': 658, 'max_candidate_release_lift': 0.75, 'next_mechanism': 'run_fold_local_weighted_bc_or_ppo'}
bc_train {'decision': 'state_objective_bc_ready_for_training', 'selected_profile': 'aux_state_head_joint', 'selected_target_scope': 'fold_local', 'selected_bc_readiness_gate': True, 'selected_label_distribution_gate': True, 'selected_fold_local_scope_gate': True, 'selected_dominant_action_

In [3]:
target_coverage = (
    joint.assign(fold_eval=joint["fold"].map(fold_eval_name))
    .groupby(["fold_eval", "fold"], dropna=False)
    .agg(
        target_rows=("datetime", "count"),
        release_gate_rows=("release_gate_state", "sum"),
        rebound_rows=("is_rebound_cluster", "sum"),
        drawdown_control_rows=("drawdown_control_state", "sum"),
        downtrend_rows=("is_downtrend_state", "sum"),
        mean_objective_weight=("objective_weight", "mean"),
        max_objective_weight=("objective_weight", "max"),
    )
    .reset_index()
)

latest_required = latest_holdout[latest_holdout["fold"].isin(REQUIRED_FOLDS)].copy()
latest_required["accuracy_gap_to_0_60"] = (MIN_SLICE_ACCURACY - pd.to_numeric(latest_required["accuracy"], errors="coerce")).clip(lower=0.0)
latest_required["sample_gate_observed"] = pd.to_numeric(latest_required["samples"], errors="coerce").fillna(0).astype(int) > 0
latest_required.to_csv(OUT_DIR / "holdout_slice_gate.csv", index=False)

holdout_by_fold = (
    latest_required.groupby("fold", dropna=False)
    .agg(
        holdout_slice_gate=("slice_gate", bool_all),
        holdout_slices=("slice", lambda s: ",".join(sorted(map(str, s)))),
        failed_slices=("slice", lambda s: ",".join(sorted(map(str, latest_required.loc[s.index][~latest_required.loc[s.index, "slice_gate"].astype(bool)]["slice"])))),
        min_holdout_accuracy=("accuracy", "min"),
        min_holdout_samples=("samples", "min"),
        max_accuracy_gap_to_0_60=("accuracy_gap_to_0_60", "max"),
    )
    .reset_index()
)

train_slice = latest_train_holdout[latest_train_holdout["target_split"].eq("train")].copy()
train_by_fold = (
    train_slice.groupby("fold", dropna=False)
    .agg(
        min_train_accuracy=("accuracy", "min"),
        min_train_samples=("samples", "min"),
        train_failed_slices=("slice", lambda s: ",".join(sorted(map(str, train_slice.loc[s.index][train_slice.loc[s.index, "accuracy"].astype(float) < MIN_SLICE_ACCURACY]["slice"])))),
    )
    .reset_index()
)

alignment = target_coverage.merge(holdout_by_fold, left_on="fold_eval", right_on="fold", how="outer", suffixes=("_target", "_holdout"))
alignment = alignment.merge(train_by_fold, left_on="fold_eval", right_on="fold", how="left", suffixes=("", "_train"))
alignment["target_side_gate"] = bool(scope_summary["training_allowed"]) and bool(lift_summary["training_allowed"]) and bool(bc_train_summary["selected_bc_readiness_gate"])
alignment["holdout_slice_gate"] = alignment["holdout_slice_gate"].fillna(False).astype(bool)
alignment["joint_target_holdout_gate"] = alignment["target_side_gate"].astype(bool) & alignment["holdout_slice_gate"].astype(bool)
alignment.to_csv(OUT_DIR / "target_holdout_alignment.csv", index=False)

overall_holdout_gate = bool(alignment[alignment["fold_eval"].isin(REQUIRED_FOLDS)]["holdout_slice_gate"].all())
overall_target_gate = bool(scope_summary["training_allowed"] and lift_summary["training_allowed"] and bc_train_summary["selected_bc_readiness_gate"])
combined_gate = bool(overall_target_gate and overall_holdout_gate)

alignment[["fold_eval", "target_rows", "release_gate_rows", "drawdown_control_rows", "downtrend_rows", "holdout_slice_gate", "failed_slices", "min_holdout_accuracy", "min_holdout_samples", "joint_target_holdout_gate"]]


,fold_eval,target_rows,release_gate_rows,drawdown_control_rows,downtrend_rows,holdout_slice_gate,failed_slices,min_holdout_accuracy,min_holdout_samples,joint_target_holdout_gate
0,wf_2020,2470,42,976,970,False,release_gate_state,0.461538,13,False
1,wf_2022,505,9,25,236,False,"drawdown_control_state,is_downtrend_state,rele...",0.000000,3,False
2,wf_2023_2024,756,11,35,480,False,"drawdown_control_state,is_downtrend_state,rele...",0.285714,3,False


In [4]:
source_selected = source_holdout_summary.get("selected_by_fold", {})
post_selected = post_short_summary.get("selected_by_fold", {})
rows = []
for fold in REQUIRED_FOLDS:
    old = source_selected.get(fold, {}) if isinstance(source_selected, dict) else {}
    new = post_selected.get(fold, {}) if isinstance(post_selected, dict) else {}
    rows.append({
        "fold": fold,
        "source_holdout_selected_candidate": old.get("selected_candidate", ""),
        "source_holdout_slice_gate": old.get("holdout_slice_gate", False),
        "source_train_slice_gate": old.get("train_slice_gate", False),
        "source_selected_relative_return": old.get("selected_relative_return"),
        "source_selected_max_dd": old.get("selected_max_dd"),
        "post_short_selected_candidate": new.get("selected_candidate", ""),
        "post_short_candidate_gate": new.get("selected_candidate_gate", False),
        "post_short_relative_return": new.get("selected_relative_return"),
        "post_short_max_dd": new.get("selected_max_dd"),
        "post_short_failure_type": new.get("failure_type", ""),
    })
reconciliation = pd.DataFrame(rows)
reconciliation["relative_return_gap_to_target_pp"] = (TARGET_RELATIVE_RETURN - pd.to_numeric(reconciliation["post_short_relative_return"], errors="coerce")).clip(lower=0.0)
reconciliation["max_dd_gap_to_target_pp"] = (pd.to_numeric(reconciliation["post_short_max_dd"], errors="coerce") - TARGET_MAX_DD).clip(lower=0.0)
reconciliation.to_csv(OUT_DIR / "source_replay_reconciliation.csv", index=False)

failed_folds = sorted(alignment.loc[alignment["fold_eval"].isin(REQUIRED_FOLDS) & ~alignment["holdout_slice_gate"], "fold_eval"].dropna().astype(str).unique())
selected_by_fold = {}
for fold in REQUIRED_FOLDS:
    rec = reconciliation[reconciliation["fold"].eq(fold)].iloc[0].to_dict()
    align_rows = alignment[alignment["fold_eval"].eq(fold)]
    align = align_rows.iloc[0].to_dict() if not align_rows.empty else {}
    post = post_selected.get(fold, {}) if isinstance(post_selected, dict) else {}
    selected_by_fold[fold] = {
        "selected_candidate": rec.get("post_short_selected_candidate", ""),
        "selected_candidate_gate": bool(rec.get("post_short_candidate_gate", False)),
        "selected_relative_return": None if pd.isna(rec.get("post_short_relative_return")) else float(rec.get("post_short_relative_return")),
        "selected_max_dd": None if pd.isna(rec.get("post_short_max_dd")) else float(rec.get("post_short_max_dd")),
        "target_rows": int(align.get("target_rows", 0) or 0),
        "release_gate_rows": int(align.get("release_gate_rows", 0) or 0),
        "drawdown_control_rows": int(align.get("drawdown_control_rows", 0) or 0),
        "holdout_slice_gate": bool(align.get("holdout_slice_gate", False)),
        "failed_slices": str(align.get("failed_slices", "")),
        "min_holdout_accuracy": None if pd.isna(align.get("min_holdout_accuracy")) else float(align.get("min_holdout_accuracy")),
        "max_dd_gap_to_target_pp": float(post.get("max_dd_gap_to_target_pp", rec.get("max_dd_gap_to_target_pp", 0.0)) or 0.0),
        "relative_gap_to_target_pp": float(post.get("relative_gap_to_target_pp", rec.get("relative_return_gap_to_target_pp", 0.0)) or 0.0),
        "failure_type": post.get("failure_type", ""),
    }

training_contract = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "role": "diagnose",
    "canonical_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "direct_comparison_baseline": "release16_post_short_training_reward_checkpoint_contract_v1",
    "source_target_experiment": "release16_joint_drawdown_release_target_contract_v1",
    "source_holdout_replay_experiment": "release16_joint_drawdown_aux_state_head_bc_holdout_replay_v1",
    "source_post_short_experiment": "release16_post_short_training_reward_checkpoint_contract_v1",
    "required_wf_windows": REQUIRED_FOLDS,
    "target_max_dd_pct": TARGET_MAX_DD,
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_side_gate": overall_target_gate,
    "target_scope_gate": bool(scope_summary["training_allowed"]),
    "target_lift_gate": bool(lift_summary["training_allowed"]),
    "train_bc_readiness_gate": bool(bc_train_summary["selected_bc_readiness_gate"]),
    "holdout_slice_gate": overall_holdout_gate,
    "target_contract_gate": combined_gate,
    "training_allowed_next": False,
    "ppo_allowed_next": False,
    "actor_retention_allowed": False,
    "next_mechanism": "repair_state_conditioned_action_head_or_aux_state_loss_before_reward_checkpoint_screen",
    "required_objective_terms": post_short_summary.get("required_objective_terms", []),
    "blocked_reason": "fold_local_target_ready_but_holdout_slice_gate_failed",
}
write_json(OUT_DIR / "training_contract.json", training_contract)

print(json.dumps(training_contract, indent=2, sort_keys=True))
reconciliation


{
  "actor_retention_allowed": false,
  "blocked_reason": "fold_local_target_ready_but_holdout_slice_gate_failed",
  "canonical_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
  "created_at": "2026-06-15T10:15:17.451991+00:00",
  "direct_comparison_baseline": "release16_post_short_training_reward_checkpoint_contract_v1",
  "experiment": "release16_joint_target_holdout_slice_repair_v1",
  "holdout_slice_gate": false,
  "next_mechanism": "repair_state_conditioned_action_head_or_aux_state_loss_before_reward_checkpoint_screen",
  "ppo_allowed_next": false,
  "required_objective_terms": [
    "checkpoint_objective_loss",
    "drawdown_drift_penalty",
    "trend_preservation_floor"
  ],
  "required_wf_windows": [
    "wf_2020",
    "wf_2022",
    "wf_2023_2024"
  ],
  "role": "diagnose",
  "source_holdout_replay_experiment": "release16_joint_drawdown_aux_state_head_bc_holdout_replay_v1",
  "source_post_short_experiment": "release16_post_short_training_reward_checkpoint_contract_v1",


,fold,source_holdout_selected_candidate,source_holdout_slice_gate,source_train_slice_gate,source_selected_relative_return,source_selected_max_dd,post_short_selected_candidate,post_short_candidate_gate,post_short_relative_return,post_short_max_dd,post_short_failure_type,relative_return_gap_to_target_pp,max_dd_gap_to_target_pp
0,wf_2020,risk_guarded_aux,False,True,-5.315300,20.932064,bc_initialization,False,-264.728894,20.918629,joint_tradeoff,294.728894,0.918629
1,wf_2022,drawdown_heavy_aux,False,True,58.109649,29.516629,bc_initialization,False,55.748206,26.832912,drawdown_control,0.000000,6.832912
2,wf_2023_2024,class_balanced_aux,False,True,29.767921,17.367905,bc_initialization,False,32.133653,22.924003,drawdown_control,0.000000,2.924003


In [5]:
worst_dd = max(float(v.get("selected_max_dd") or 0.0) for v in selected_by_fold.values())
min_relative = min(float(v.get("selected_relative_return") or 0.0) for v in selected_by_fold.values())
max_dd_gap = max(float(v.get("max_dd_gap_to_target_pp") or 0.0) for v in selected_by_fold.values())
max_relative_gap = max(float(v.get("relative_gap_to_target_pp") or 0.0) for v in selected_by_fold.values())

summary = {
    "experiment": EXPERIMENT,
    "created_at": training_contract["created_at"],
    "decision": "release16_joint_target_holdout_slice_repair_contract_no_training",
    "role": "diagnose",
    "changed_layer": ["target", "model_selection", "bc_to_ppo_handoff"],
    "canonical_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "direct_comparison_baseline": "release16_post_short_training_reward_checkpoint_contract_v1",
    "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "source_experiment": "release16_post_short_training_reward_checkpoint_contract_v1",
    "source_target_experiment": "release16_joint_drawdown_release_target_contract_v1",
    "source_holdout_replay_experiment": "release16_joint_drawdown_aux_state_head_bc_holdout_replay_v1",
    "source_target_decision": source_summary.get("decision"),
    "source_holdout_replay_decision": source_holdout_summary.get("decision"),
    "source_post_short_decision": post_short_summary.get("decision"),
    "target_side_gate": overall_target_gate,
    "target_scope_gate": bool(scope_summary["training_allowed"]),
    "target_lift_gate": bool(lift_summary["training_allowed"]),
    "train_bc_readiness_gate": bool(bc_train_summary["selected_bc_readiness_gate"]),
    "bc_readiness_gate": False,
    "bc_holdout_slice_gate": overall_holdout_gate,
    "target_contract_gate": combined_gate,
    "bc_screen_allowed_next": False,
    "training_allowed_next": False,
    "next_training_allowed_after_target_revision": False,
    "ppo_allowed_next": False,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "primary_blocker": "bc_candidate_or_slice_gate_failed",
    "next_action": "repair_state_conditioned_action_head_or_aux_state_loss_before_reward_checkpoint_screen",
    "next_mechanism": "repair_state_conditioned_action_head_or_aux_state_loss_before_reward_checkpoint_screen",
    "required_objective_terms": training_contract["required_objective_terms"],
    "required_wf_windows": REQUIRED_FOLDS,
    "failed_folds": failed_folds,
    "selected_by_fold": selected_by_fold,
    "worst_wf_max_drawdown_pct": worst_dd,
    "min_wf_relative_return_pp": min_relative,
    "max_drawdown_gap_to_target_pp": max_dd_gap,
    "max_relative_gap_to_target_pp": max_relative_gap,
    "target_max_dd_pct": TARGET_MAX_DD,
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "sidecars": [
        "target_scope_audit.csv",
        "target_lift_audit.csv",
        "bc_readiness_scores.csv",
        "target_contract_gate.csv",
        "holdout_slice_gate.csv",
        "target_holdout_alignment.csv",
        "source_replay_reconciliation.csv",
        "training_contract.json",
        "summary.json",
        "conclusion.md",
    ],
}
write_json(OUT_DIR / "summary.json", summary)

conclusion = f"""# Release16 Joint Target Holdout Slice Repair

Decision: `{summary['decision']}`.

The fold-local joint target contract is still target-side ready: scope gate `{summary['target_scope_gate']}`, lift gate `{summary['target_lift_gate']}`, and train label readiness `{summary['train_bc_readiness_gate']}`. That does not authorize training because the latest post-short holdout slice gate is `{summary['bc_holdout_slice_gate']}` across required folds `{', '.join(REQUIRED_FOLDS)}`.

Worst selected MaxDD remains `{summary['worst_wf_max_drawdown_pct']:.2f}%` versus target `< {TARGET_MAX_DD:.2f}%`, and minimum selected relative return remains `{summary['min_wf_relative_return_pp']:.2f}pp` versus target `>= {TARGET_RELATIVE_RETURN:.2f}pp`. Failed folds: `{', '.join(failed_folds)}`.

No actor is retained. No PPO or full reward-checkpoint replay should start from this contract until a state-conditioned action head, auxiliary state loss, or equivalent target enrichment passes the holdout slice gate before official three-fold replay.
"""
(OUT_DIR / "conclusion.md").write_text(conclusion, encoding="utf-8")

display(Markdown(conclusion))
summary


# Release16 Joint Target Holdout Slice Repair

Decision: `release16_joint_target_holdout_slice_repair_contract_no_training`.

The fold-local joint target contract is still target-side ready: scope gate `True`, lift gate `True`, and train label readiness `True`. That does not authorize training because the latest post-short holdout slice gate is `False` across required folds `wf_2020, wf_2022, wf_2023_2024`.

Worst selected MaxDD remains `26.83%` versus target `< 20.00%`, and minimum selected relative return remains `-264.73pp` versus target `>= 30.00pp`. Failed folds: `wf_2020, wf_2022, wf_2023_2024`.

No actor is retained. No PPO or full reward-checkpoint replay should start from this contract until a state-conditioned action head, auxiliary state loss, or equivalent target enrichment passes the holdout slice gate before official three-fold replay.


{'experiment': 'release16_joint_target_holdout_slice_repair_v1',
 'created_at': '2026-06-15T10:15:17.451991+00:00',
 'decision': 'release16_joint_target_holdout_slice_repair_contract_no_training',
 'role': 'diagnose',
 'changed_layer': ['target', 'model_selection', 'bc_to_ppo_handoff'],
 'canonical_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
 'direct_comparison_baseline': 'release16_post_short_training_reward_checkpoint_contract_v1',
 'reference_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
 'source_experiment': 'release16_post_short_training_reward_checkpoint_contract_v1',
 'source_target_experiment': 'release16_joint_drawdown_release_target_contract_v1',
 'source_holdout_replay_experiment': 'release16_joint_drawdown_aux_state_head_bc_holdout_replay_v1',
 'source_target_decision': 'release16_joint_drawdown_aux_state_head_target_contract_ready_for_bc_screen_no_promotion',
 'source_holdout_replay_decision': 'release16_joint_drawdown_aux_state_head_bc_holdout_slice_